<a href="https://colab.research.google.com/github/c-lydia/gpt/blob/main/gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT Transformer Training
---

## What is a Transformer?

A **transformer** is a neural network that learns patterms in sequences of data (like text). Models like RNNS (Recurrent Neural Network) processes text word-by-word, left to right (slow). The transformer processes all words simulatneously using **self-attention** (fast and powerful).

Imagine you're reading a book:

- **RNN:** read word-by-word, remember previous words in short-term memoy
- **Transformer:** look at the entire sentence at once, figure out which words are important to understand each word

Self-attention asnwers the question: When processing a word, which other words should I pay attention to?

In the sentence "The bank executive was fired", when processing "fired":

- High attention to "executive" and "bank" (who was fired)
- Low attention to "the" (not important)
- Medium attention to "was" (helps understand the action)

The model learns this automatically.

---

## Tokenizer

A **tokenizer** converts text into **numbers** that the neural network can process. Our tokenier works at the **character level** (not word level).

---

### Build Vocabulary

``` python
text = 'hello world'
chars = sorted(set(text))

char_to_idx = {c: i for i, c in enumerate(chars)}
```

What it does:

- Finds all unique characters in text
- Sorts them alphabetically
- Assigned each character a unique number

**Vocab size:** how many unique characters (7 in this example)

---

### Encoder (text to numbers)

``` python
def encode(self, text):
  return torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long)
```

For text "hello":

``` txt
'h' → char_to_idx['h'] → 3
'e' → char_to_idx['e'] → 2
'l' → char_to_idx['l'] → 4
'l' → char_to_idx['l'] → 4
'o' → char_to_idx['o'] → 5

Result: tensor([3, 2, 4, 4, 5])
```

Why `torch.tensor`? PyTorch models need PyTorch tensors, not Python lists.

Why `dtype = torch.long`? Stores as 64-bit integers (standard for indices).

---

### Decode (numbers to text)

``` python
def decode(self, token_ids):
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.tolist()
    return ''.join([self.idx_to_char[i] for i in token_ids])
```

For tensor `[3, 2, 4, 4, 5]`:

``` txt
3 → idx_to_char[3] → 'h'
2 → idx_to_char[2] → 'e'
4 → idx_to_char[4] → 'l'
4 → idx_to_char[4] → 'l'
5 → idx_to_char[5] → 'o'

Result: "hello"
```

**Round-trip**: text is encoded to numbers, then numbers are decoded to text.

---

## Model Architecture

---
### The big picture

``` txt
Input (text)
    ↓
Tokenize (text → numbers)
    ↓
Embedding Layer (add meaning)
    ↓
+ Positional Encoding (add position info)
    ↓
Transformer Blocks × N (learn patterns)
    ↓
Output Layer (project to vocab)
    ↓
Logits (probabilities for next token)
```

---

### Embedding layer

``` python
self.embedding = nn.Embedding(vocab_size, d_model)
```

What it does:

- Takes a number (token ID) like `3`
- Converts it to a vector of `d_model` dimensions
- These vectors are learned during training

**Example:**

``` txt
Input token ID: 3
↓
Embedding: [0.2, -0.5, 0.8, ..., 0.1]  (128 numbers for d_model=128)
```

Why? Numbers don't have meaning, embedding vectors learn what each character/word means.

**Learning:** during training, these embeddings are adjusted so similar tokens have similar vectors.

---

### Positioning encoding

``` python
pe[:, 0::2] = torch.sin(pos * div_term)
pe[:, 1::2] = torch.cos(pos * div_term)
```

**Problem:** transformers process all tokens in parallel, so they don't naturally know positions.

**Solution:** add position information using sine and cosine waves.

**Why sine/cosine?**

- Difference frequemncies encode different scales (nearby vs far position)
- Model learns to extract relative positions automatically
- Works for any sequence length

**Example:**

``` txt
Position 0: [sin(0), cos(0), sin(0), cos(0), ...]
Position 1: [sin(1), cos(1), sin(1), cos(1), ...]
Position 2: [sin(2), cos(2), sin(2), cos(2), ...]
```

Added ro embedding:

``` txt
Token embedding: [0.2, -0.5, 0.8, ...]
+ Position encoding: [0.0, 1.0, 0.84, ...]
= Final input: [0.2, 0.5, 1.68, ...]
```

---

### Self-Attention

For each word, which other words are important to understand it?

**Process:**

1. **Create Queries (Q):** What am I looking for?
2. **Create Keys (K):** What am I offering?
3. **Create Values (V):** What's my information?
4. **Compare:** $Q \cdot K^{T}$, How much does query match key?
5. **Softmax:** Convert to porbabilities $(0-1)$
6. **Extract:** Multiply by values to get important information

**Code breakdown:**

``` python
Q = self.W_q(x)  # Project input to queries
K = self.W_k(x)  # Project input to keys
V = self.W_v(x)  # Project input to values

scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
# scores[i,j] = how much token i should attend to token j

attn_weights = F.softmax(scores, dim=-1)
# Convert scores to probabilities (sum to 1)

context = torch.matmul(attn_weights, V)
# Weighted sum: take important parts from V
```

**Example:**

Processing word "it" in "The cast sat on the mat because it was soft"

``` txt
Q (for "it"): [0.1, 0.2, 0.3, ...]  "looking for subject"
K (for each word):
  "the": [0.05, 0.1, ...]
  "cat": [0.15, 0.25, ...]  ← Matches well!
  "sat": [0.08, 0.12, ...]
  "mat": [0.16, 0.26, ...]  ← Matches well!
  ...

Q·K^T gives high scores for "cat" and "mat"
Softmax: attention weights ≈ [0.05, 0.4, 0.1, 0.35, ...]
         "it" mostly attends to "cat" and "mat"
```

---

### Multi-Head Attention

**Problem:** One attention mechanism might only find one type of relationship.

**Solution:** Use multiple attention heads in parallel.

``` python
self.transformer_blocks = nn.ModuleList([
    TransformerBlock(d_model, num_heads) for _ in range(num_layers)
])
```

Each head focuses on different relationships:

- Head 1: Subject-object relationships
- Head 2: Verb-noun relationships
- Head 3: Adjective-noun relationships
- Head 4: Long-range dependencies

They work in parallel, then concatenate results.

---

### Feed-Forward Network

``` python
nn.Linear(d_model, 4 * d_model)  # Expand
nn.ReLU()                          # Non-linearity
nn.Linear(4 * d_model, d_model)    # Contract
```

What it does:

- Expands representation to 4× size
- Applies ReLU (activation function)
- Contracts back to original size

Why? Learns transformations beyond what attention can do. Different for each position.

---

### Layer Normalization

``` python
x = x + self.attn(self.ln1(x), mask)
x = x + self.ffn(self.ln2(x))
```

What `ln1` does:

- Normalize activations to mean 0, variance 1
- Helps training stability
- Appp=lied before attention

What `+` does:

- **Residual connection:** directly pass input + output
- Helps gradients flow backward
- Prevents vanishing gradients in deep networks

---

### Causal Mask

``` python
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
scores = scores.masked_fill(mask, float('-inf'))
```

**Problem:** In language modeling, can't look at future tokens (cheating!).

**Solution:** Set future token scores to $-\infty$ (so softmax ≈ 0).

**Example:** Processing position 2 (can only see positions 0, 1):

``` txt
Position: 0  1  2  3  4
Can see:  ✓  ✓  ✓  ✗  ✗
Mask:     0  0  0 -∞ -∞
After softmax: [0.4, 0.3, 0.3, 0.0, 0.0]
```

---

### Output Layer

``` python
self.lm_head = nn.Linear(d_model, vocab_size)
```

What it does:

- Takes d_model-dimensional representation
- Projects to vocab_size probabilities
- For next token prediction

**Output:**

``` txt
Input: [0.2, -0.5, 0.8, ...]  (128 dims)
↓
Linear transformation
↓
Output: [0.1, 0.05, 0.3, ...]  (100 dims for vocab_size=100)
```

---

## Training Process

---

### Prepare data

``` python
ids = tokenizer.encode(text_data)
split_idx = int(0.9 * len(ids))
train_ids = ids[:split_idx]
val_ids = ids[split_idx:]
```

What it does:

- Converts all text to token IDs
- Splits into 90% training, 10% validation

---

### Create batches

``` python
def get_batch(ids, batch_size, seq_len, device):
    max_start = len(ids) - seq_len
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([ids[i:i+seq_len] for i in starts])
    y = torch.stack([ids[i+1:i+seq_len+1] for i in starts])
    return x.to(device), y.to(device)
```

What it does:

- Randomly pick `batch_size` starting positions
- Extract `seq_len` tokens starting at each position
- Shift by 1 (predict next token)

**Example:**

``` txt
ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
start = 2, seq_len = 3

x = [2, 3, 4]     (input)
y = [3, 4, 5]     (target - what comes next)

Model learns: given [2,3,4], predict [3,4,5]
```

---

### Forward pass

``` python
logits = model(x)  # (batch_size, seq_len, vocab_size)
```

What it does:

- Pass input through all layers
- Output probabilities for each next token

**Shape example:**

``` txt
Input x: (32, 128)           batch_size=32, seq_len=128
↓
Embedding: (32, 128, 128)    128 dims per token
↓
Attention & FFN: (32, 128, 128)
↓
Output: (32, 128, 100)       vocab_size=100
```

---

### Calculate loss

``` python
loss = nn.CrossEntropyLoss()(logits.view(-1, vocab_size), y.view(-1))
```

What `CrossEntropyLoss` does:

- Compares predicted probabilities with true next token
- Returns a single number (lower = better predictions)

How it works:

``` txt
True: [3, 4, 5]  (actual next tokens)
Predicted: [[0.1, 0.05, 0.3, 0.55], ...]  (probabilities)

Loss = -log(P(3|context)) - log(P(4|context)) - log(P(5|context))
     = -log(0.55) - log(0.5) - log(0.6)  (example)
     ≈ 1.8  (lower is better)
```

---

### Backward pass

``` python
optimizer.zero_grad()
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
```

Step by step:

1. `zero_grad()`: clear old gradients
2. `loss.backward()`: compute gradients using backpropagation
3. `clip_grad_norm()`: prevent exploding gradients
4. `optimizer_step()`: update weights using gradients

What happens:

``` txt
Loss: 1.8
↓ Backprop
Compute how much each weight contributes to loss
↓ Update
Decrease weights that caused high loss
Increase weights that caused low loss
↓
New loss: 1.7 (improved!)
```

---

### Repeat

Do step 3-5 for hundreds/throusands of batches unile:

- Training loss is low
- Validation loss stops improving (stop overfitting)

---

## Key concepts

---

### Gradient Descent

**Goal:** Make the model better by adjusting weights.

**How:**

1. Compute loss (how wrong we are)
2. Compute gradient (which direction to move)
3. Move weights in that direction
4. Repeat until loss is minimal

**Analogy:** Walking down a hill blindfolded. Gradient tells you which way is downhill.
---

### Learning Rate

``` python
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
```

**What it controls:** How much to change weights each step.

Too high (`lr=0.1`):

- Changes too much
- Loss bounces around wildly
- Might never converge

Too low (`lr=1e-6`):

- Changes too little
- Training is very slow
- Might get stuck

Just right (`lr=1e-3`):

- Loss decreases smoothly
- Converges reasonably fast

---

### Overfitting

**Problem:** Model memorizes training data instead of learning patterns.

**Signs:**

- Training loss: ↓ (keeps decreasing)
- Validation loss: ↑ (starts increasing)

**Solution:**

- Use less complex model
- Add dropout (randomly disable neurons)
- Add regularization (weight decay)
- Train for fewer epochs

---

### Batch Size

``` python
batch_size = 32
```

**What it is:** Process 32 sequences at once before updating weights.

**Tradeoff:**

- Larger batch: More stable, faster per sample, uses more memory
- Smaller batch: Noisier gradients, regularization effect

---

### Sequence Length

``` python
seq_len = 128
```

**What it is:** Each sequence is 128 tokens long.

**Tradeoff:**

- Longer: Model sees more context, slower (attention is $O(n^{2})$)
- Shorter: Fast, but less context

---

### Epochs

``` python
for epoch in range(num_epochs):
```

**What it is:** One complete pass through all training data.

**Example:** If you have 1M tokens and `batch_size=32, seq_len=128`:

- One epoch = pass through all 1M tokens
- One epoch = ~250 gradient updates

---

## Why It Works

---

### Self-Attention is Powerful

- Can compare any position to any other position
- Learns to focus on relevant information
- Parallelizable (all positions at once)

---

### Layer Stacking

- Multiple layers learn hierarchical patterns
- First layers: Local patterns (characters, short phrases)
- Middle layers: Semantic patterns (meaning)
- Last layers: Long-range patterns (story structure)

---

### Residual Connections

- Allow gradients to flow directly through layers
- Prevent vanishing gradients in deep networks
- Easier training

---

### Layer Normalization

- Stabilizes training
- Consistent scales across layers

---

### Causal Masking

- Prevents cheating (looking at future)
- Makes it a true language model

# Implementation

---

## Device configuration

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
import urllib.request
from tqdm import tqdm
import os

if torch.cuda.is_available():
  device = torch.device('cuda')
else:
  device = torch.device('cpu')

print(f'Use device: {device}')

if device.type == 'cuda':
  print(f'GPU: {torch.cuda.get_device_name()}')
  print(f'VRAM: {torch.cuda.get_device_properties(device).total_memory / 1e9:.1f}GB')




Use device: cpu


---

## Download data


In [2]:
def download_gutenberg_books(num_books = 5):
  books = {
        11: "Alice in Wonderland",
        25: "Peter Pan",
        35: "The Time Machine",
        36: "The Island of Doctor Moreau",
        37: "The Invisible Man",
        46: "A Christmas Carol",
        74: "The Adventures of Sherlock Holmes",
        98: "A Tale of Two Cities",
        141: "The Adventures of Pinocchio",
        145: "Robinson Crusoe",
        408: "Jane Eyre",
        514: "Little Women",
        524: "Frankenstein",
        612: "The Odyssey",
        1232: "The Prince",
        1268: "The Count of Monte Cristo",
        1342: "Pride and Prejudice",
        1400: "Great Expectations",
        1513: "Romeo and Juliet",
        1524: "Hamlet",
        1534: "Macbeth",
        1642: "Sense and Sensibility",
        1661: "Sherlock Holmes: A Study in Scarlet",
        1727: "The Count of Monte Cristo",
        2554: "Crime and Punishment",
        2600: "War and Peace",
        2701: "Moby Dick",
        2814: "Anna Karenina",
        3825: "The Iliad",
        4085: "The Great Gatsby",
        4300: "Ulysses",
        5200: "Metamorphosis",
        5230: "The Scarlet Letter",
        5976: "The War of the Worlds",
        6130: "The Jungle Book",
        6151: "The Secret Garden",
        11844: "David Copperfield",
        33283: "The Awakening",
        84: "Frankenstein",
        768: "Wuthering Heights",
        944: "Treasure Island",
        1111: "Vanity Fair",
        1184: "The Iliad",
        1220: "Ivanhoe",
        1254: "The Yellow Wallpaper",
        1500: "A Midsummer Night's Dream",
        1501: "A Doll's House",
        1661: "The Adventures of Sherlock Holmes",
        1952: "The Hound of the Baskervilles"
  }

  all_text = ''

  for book_id in list(books.keys())[:num_books]:
    title = books[book_id]
    url = f'https://www.gutenberg.org/ebooks/{book_id}.txt.utf-8'

    try:
      print(f'Downloading: {title}...', end = '')

      with urllib.request.urlopen(url, timeout = 10) as response:
        text = response.read().decode('utf-8')

      if '***' in text:
        text = text.split('***')[2]

      all_text += text + '\n\n'
      size_kb = len(text) / 1024
      print(f'size: {size_kb:.0f}KB')
    except Exception as e:
      print(f'Error downloading: {e}')

  return all_text

print('Downloading text data...\n')
text_data = download_gutenberg_books(num_books = 50)
print(f'Total downloaded text length: {len(text_data)} characters')


Downloading: Alice in Wonderland...size: 145KB
Downloading: Peter Pan...size: 2KB
Downloading: The Time Machine...size: 179KB
Downloading: The Island of Doctor Moreau...size: 336KB
Downloading: The Invisible Man...size: 4KB
Downloading: A Christmas Carol...size: 158KB
Downloading: The Adventures of Sherlock Holmes...size: 392KB
Downloading: A Tale of Two Cities...size: 641KB
Downloading: The Adventures of Pinocchio...size: 879KB
Downloading: Robinson Crusoe...size: 1771KB
Downloading: Jane Eyre...size: 64KB
Downloading: Little Women...size: 1004KB
Downloading: Frankenstein...size: 547KB
Downloading: The Odyssey...size: 32KB
Downloading: The Prince...size: 281KB
Downloading: The Count of Monte Cristo...size: 1109KB
Downloading: Pride and Prejudice...size: 726KB
Downloading: Great Expectations...size: 991KB
Downloading: Romeo and Juliet...size: 144KB
Downloading: Hamlet...size: 180KB
Downloading: Macbeth...size: 155KB
Downloading: Sense and Sensibility...size: 50KB
Downloading: The Adve

---

## `CharTokenizer`

In [20]:
class CharTokenizer:
  def __init__(self, text):
    self.chars = sorted(list(set(text)))
    self.vocab_size = len(self.chars)

    self.char_to_idx = {char: i for i, char in enumerate(self.chars)}
    self.idx_to_char = {i: char for i, char in enumerate(self.chars)}

  def encode(self, text):
    result = []

    for c in text:
      token_id = self.char_to_idx.get(c, self.char_to_idx['?'])
      result.append(token_id)
    return torch.tensor(result, dtype = torch.long)

  def decode(self, token_ids):
    if isinstance(token_ids, torch.Tensor):
      token_ids = token_ids.tolist()

    characters = []

    for i in token_ids:
      char = self.idx_to_char[i]
      characters.append(char)

    return ''.join(characters)

tokenizer = CharTokenizer(text_data) # Re-initialize tokenizer with text_data

ids = tokenizer.encode(text_data)
print(f'\nEncoded: {ids.shape[0]:,} tokens')

split_idx = int(0.9 * len(ids))
train_ids = ids[:split_idx]
val_ids = ids[split_idx:]
print(f'Train: {len(train_ids):,}, val: {len(val_ids):,}')


Encoded: 28,219,765 tokens
Train: 25,397,788, val: 2,821,977


---

## `MultiHeadAttention`

In [4]:
from torch.nn.modules.activation import MultiheadAttention
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads, dropout = 0.1):
    super().__init__()
    assert d_model % num_heads == 0

    self.d_model = d_model
    self.num_heads = num_heads
    self.d_k = d_model // num_heads

    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)
    self.W_o = nn.Linear(d_model, d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, mask = None):
    batch_size, seq_len, _ = x.shape

    Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
    K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
    V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)

    if mask is not None:
      scores = scores.masked_fill(mask, float('-inf'))

    atten_weight = F.softmax(scores, dim = -1)
    atten_weight = self.dropout(atten_weight)
    context = torch.matmul(atten_weight, V)

    context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
    output = self.W_o(context)
    return output

class FeedForward(nn.Module):
  def __init__(self, d_model, dropout = 0.1):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(d_model, 4 * d_model),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(4 * d_model, d_model),
        nn.Dropout(dropout)
    )

  def forward(self, x):
    return self.net(x)

class TransformerBlock(nn.Module):
  def __init__(self, d_model, num_heads, dropout = 0.1):
    super().__init__()
    self.attn = MultiHeadAttention(d_model, num_heads, dropout)
    self.ffn = FeedForward(d_model, dropout)
    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)

  def forward(self, x, mask = None):
    x = x + self.attn(self.ln1(x), mask)
    x = x + self.ffn(self.ln2(x))
    return x

class GPT(nn.Module):
  def __init__(self, vocab_size, d_model = 128, num_heads = 4, num_layers = 4, max_seq_len = 512, dropout = 0.1):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, d_model)

    pe = torch.zeros(max_seq_len, d_model)
    pos = torch.arange(0, max_seq_len, dtype = torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0))/d_model))
    pe[:, 0::2] = torch.sin(pos * div_term)
    pe[:, 1::2] = torch.cos(pos * div_term)
    self.register_buffer('pos_encoding', pe)

    transformer_blocks = []

    for layer_num in range(num_layers):
      block = TransformerBlock(d_model, num_heads, dropout)
      transformer_blocks.append(block)

    self.transformer_blocks = nn.ModuleList(transformer_blocks)

    self.ln_final = nn.LayerNorm(d_model)
    self.lm_head = nn.Linear(d_model, vocab_size)

    self.d_model = d_model
    self.vocab_size = vocab_size

  def _get_causal_mask(self, seq_len, device):
    mask = torch.triu(torch.ones(seq_len, seq_len, device = device), diagonal = 1).bool()
    return mask.unsqueeze(0).unsqueeze(0)

  def forward(self, token_ids):
    batch_size, seq_len = token_ids.shape
    device = token_ids.device

    x = self.embedding(token_ids) + self.pos_encoding[:seq_len]
    mask = self._get_causal_mask(seq_len, device)

    for block in self.transformer_blocks:
      x = block(x, mask)

    x = self.ln_final(x)
    logits = self.lm_head(x)
    return logits

print('Creating GPT model...')
model = GPT(
    vocab_size = tokenizer.vocab_size,
    d_model = 128,
    num_heads = 4,
    num_layers = 4,
    dropout = 0.1
).to(device)

total = 0

for p in model.parameters():
    num_elements = p.numel()
    total = total + num_elements

num_params = total

print(f"Model created: {num_params:,} parameters")

Creating GPT model...
Model created: 854,510 parameters


---

## Training configuration

In [5]:
def get_batch(ids, batch_size, seq_len, device):
  max_start = len(ids) - seq_len
  starts = torch.randint(0, max_start, (batch_size,))

  x_list = []

  for i in starts:
      sequence = ids[i:i+seq_len]
      x_list.append(sequence)

  x = torch.stack(x_list)

  y_list = []

  for i in starts:
      sequence = ids[i+1:i+seq_len+1]
      y_list.append(sequence)

  y = torch.stack(y_list)
  return x.to(device), y.to(device)

batch_size = 32
seq_len = 128
learning_rate = 1e-3
num_epochs = 100
eval_interval = 50

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate, weight_decay = 1e-4)
schdulaer = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = num_epochs)
loss_fn = nn.CrossEntropyLoss()

print(f'Training setup complete')
print(f'Batch size: {batch_size}')
print(f'Sequencee length: {seq_len}')
print(f'Learnign rate: {learning_rate}')
print(f'Epochs: {num_epochs}')

Training setup complete
Batch size: 32
Sequencee length: 128
Learnign rate: 0.001
Epochs: 100


---

## Training

In [6]:
print("\n" + "="*60)
print("TRAINING")
print("="*60)

best_val_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    num_batches = 0

    pbar = tqdm(range(100), desc=f"Epoch {epoch+1}/{num_epochs}")

    for step in pbar:
        x, y = get_batch(train_ids, batch_size, seq_len, device)

        logits = model(x)
        loss = loss_fn(logits.view(-1, tokenizer.vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_train_loss += loss.item()
        num_batches += 1

        if (step + 1) % eval_interval == 0:
            model.eval()
            with torch.no_grad():
                val_x, val_y = get_batch(val_ids, batch_size, seq_len, device)
                val_logits = model(val_x)
                val_loss = loss_fn(val_logits.view(-1, tokenizer.vocab_size), val_y.view(-1))

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), 'best_model.pt')

            pbar.set_postfix({
                'train_loss': f'{loss.item():.4f}',
                'val_loss': f'{val_loss.item():.4f}'
            })
            model.train()

    avg_loss = total_train_loss / num_batches
    schdulaer.step()
    print(f"Epoch {epoch+1}: Avg Loss = {avg_loss:.4f}, Best Val Loss = {best_val_loss:.4f}")

print("\n Training complete!")
print(f"Best model saved (val_loss: {best_val_loss:.4f})")


TRAINING


Epoch 1/100: 100%|██████████| 100/100 [01:38<00:00,  1.01it/s, train_loss=2.5175, val_loss=2.4445]


Epoch 1: Avg Loss = 2.7923, Best Val Loss = 2.4445


Epoch 2/100: 100%|██████████| 100/100 [01:39<00:00,  1.01it/s, train_loss=2.3586, val_loss=2.3466]


Epoch 2: Avg Loss = 2.4418, Best Val Loss = 2.3466


...

Epoch 98: Avg Loss = 1.4421, Best Val Loss = 1.3225


Epoch 99/100: 100%|██████████| 100/100 [01:38<00:00,  1.02it/s, train_loss=1.3994, val_loss=1.4295]


Epoch 99: Avg Loss = 1.4404, Best Val Loss = 1.3225


Epoch 100/100: 100%|██████████| 100/100 [01:39<00:00,  1.01it/s, train_loss=1.4364, val_loss=1.3893]

Epoch 100: Avg Loss = 1.4393, Best Val Loss = 1.3225

 Training complete!
Best model saved (val_loss: 1.3225)


---

## Text generation

In [7]:
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

print("\n" + "="*60)
print("TEXT GENERATION")
print("="*60)

@torch.no_grad()
def generate(prompt, max_tokens=100, temperature=0.8):
    """Generate text autoregressively"""
    tokens = tokenizer.encode(prompt).unsqueeze(0).to(device)

    for _ in range(max_tokens):
        logits = model(tokens)
        next_logits = logits[0, -1, :] / temperature
        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        tokens = torch.cat([tokens, next_token.unsqueeze(0)], dim=1)

    return tokenizer.decode(tokens[0].cpu())

prompts = ["The", "Once upon a time", "It was", "In the"]

for prompt in prompts:
    print(f"\n Prompt: '{prompt}'")
    generated = generate(prompt, max_tokens=100, temperature=0.8)
    print(f"Generated:\n{generated}\n")


TEXT GENERATION

 Prompt: 'The'
Generated:
The place, but say to himself among his black.

The room, was two speak communication to the bask, wh


 Prompt: 'Once upon a time'
Generated:
Once upon a time and such a notion
anotal twost of the revolice of the own in the door which was being
passed at t


 Prompt: 'It was'
Generated:
It was the far of that’s preparate
from the air and among the persuade close of the faced and into the en


 Prompt: 'In the'
Generated:
In the subtain of the bed your croding of employ,
and of a merlance in what he said, and out that she had



---

## Save model

In [8]:
torch.save(model.state_dict(), 'gpt_model.pt')

import json
tokenizer_config = {
    'chars': tokenizer.chars,
    'vocab_size': tokenizer.vocab_size,
}
with open('tokenizer_config.json', 'w') as f:
    json.dump(tokenizer_config, f)

print("Model saved to 'gpt_model.pt'")
print("Tokenizer saved to 'tokenizer_config.json'")

from google.colab import files
print("\nDownloading files to your computer...")
files.download('gpt_model.pt')
files.download('tokenizer_config.json')
print("Files downloaded!")

Model saved to 'gpt_model.pt'
Tokenizer saved to 'tokenizer_config.json'



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Files downloaded!


---

# Test

---

## Tokenizer

In [9]:
import torch

class CharTokenizer:
    def __init__(self, text):
        self.chars = sorted(set(text))
        self.vocab_size = len(self.chars)
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.idx_to_char = {i: c for c, i in self.char_to_idx.items()}

    def encode(self, text):
        return torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long)

    def decode(self, token_ids):
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.tolist()
        return ''.join([self.idx_to_char[i] for i in token_ids])

print("TEST 1: Tokenizer")
print("-" * 50)

tokenizer = CharTokenizer("hello world")
print(f"Vocab: {tokenizer.chars}")
print(f"Vocab size: {tokenizer.vocab_size}")

text = "hello"
encoded = tokenizer.encode(text)
print(f"\nEncode '{text}':")
print(f"  Result: {encoded}")
print(f"  Shape: {encoded.shape}")

decoded = tokenizer.decode(encoded)
print(f"\nDecode {encoded.tolist()}:")
print(f"  Result: '{decoded}'")

matches = (text == decoded)
print(f"\nRound-trip match: {matches}")
assert matches, "FAILED: encode/decode mismatch"
print("PASSED: Tokenizer works!")

TEST 1: Tokenizer
--------------------------------------------------
Vocab: [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
Vocab size: 8

Encode 'hello':
  Result: tensor([3, 2, 4, 4, 5])
  Shape: torch.Size([5])

Decode [3, 2, 4, 4, 5]:
  Result: 'hello'

Round-trip match: True
PASSED: Tokenizer works!


---

## Batch generation

In [10]:
import torch

def get_batch(ids, batch_size, seq_len, device):
    """Create random batch"""
    max_start = len(ids) - seq_len
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([ids[i:i+seq_len] for i in starts])
    y = torch.stack([ids[i+1:i+seq_len+1] for i in starts])
    return x.to(device), y.to(device)

print("TEST 2: get_batch Function")
print("-" * 50)

device = 'cpu'
ids = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
batch_size = 2
seq_len = 3

x, y = get_batch(ids, batch_size, seq_len, device)

print(f"x shape: {x.shape}")
print(f"y shape: {y.shape}")
print(f"\nx (input):\n{x}")
print(f"\ny (target):\n{y}")

print(f"\nVerify shift by 1:")
for i in range(batch_size):
    print(f"  Batch {i}: x={x[i].tolist()}, y={y[i].tolist()}")
    expected_sequence_list = x[i][1:].tolist() + [x[i][-1].item() + 1]
    expected_sequence_tensor = torch.tensor(expected_sequence_list, device=device)
    assert (y[i] == expected_sequence_tensor).all().item(), f"FAILED: Batch {i} shift by 1 mismatch"

print("PASSED: get_batch works!")

TEST 2: get_batch Function
--------------------------------------------------
x shape: torch.Size([2, 3])
y shape: torch.Size([2, 3])

x (input):
tensor([[0, 1, 2],
        [2, 3, 4]])

y (target):
tensor([[1, 2, 3],
        [3, 4, 5]])

Verify shift by 1:
  Batch 0: x=[0, 1, 2], y=[1, 2, 3]
  Batch 1: x=[2, 3, 4], y=[3, 4, 5]
PASSED: get_batch works!


---

## Model creation

In [11]:
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(context)
        return output


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_heads=4, num_layers=4, max_seq_len=512, dropout=0.1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        pe = torch.zeros(max_seq_len, d_model)
        pos = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        self.register_buffer('pos_encoding', pe)

        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.d_model = d_model
        self.vocab_size = vocab_size

    def _get_causal_mask(self, seq_len, device):
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
        return mask.unsqueeze(0).unsqueeze(0)

    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        device = token_ids.device

        x = self.embedding(token_ids) + self.pos_encoding[:seq_len]
        mask = self._get_causal_mask(seq_len, device)

        for block in self.transformer_blocks:
            x = block(x, mask)

        x = self.ln_final(x)
        logits = self.lm_head(x)
        return logits

print("TEST 3: Model Creation")
print("-" * 50)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab_size = 100

model = GPT(vocab_size=vocab_size, d_model=64, num_heads=4, num_layers=2).to(device)
num_params = sum(p.numel() for p in model.parameters())

print(f"Device: {device}")
print(f"Vocab size: {vocab_size}")
print(f"Model parameters: {num_params:,}")
print(f"PASSED: Model created!")

TEST 3: Model Creation
--------------------------------------------------
Device: cpu
Vocab size: 100
Model parameters: 112,996
PASSED: Model created!


---

## Forward pass

In [12]:
print("TEST 4: Forward Pass")
print("-" * 50)

batch_size = 4
seq_len = 32
x = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)

print(f"Input shape: {x.shape}")

logits = model(x)

print(f"Output shape: {logits.shape}")
print(f"Expected: ({batch_size}, {seq_len}, {vocab_size})")

assert logits.shape == (batch_size, seq_len, vocab_size), "Shape mismatch!"
assert not torch.isnan(logits).any(), "NaN in output!"

print("PASSED: Forward pass works!")

TEST 4: Forward Pass
--------------------------------------------------
Input shape: torch.Size([4, 32])
Output shape: torch.Size([4, 32, 100])
Expected: (4, 32, 100)
PASSED: Forward pass works!


---

## Backward pass

In [13]:
print("TEST 5: Backward Pass")
print("-" * 50)

model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

x = torch.randint(0, vocab_size, (4, 32)).to(device)
y = torch.randint(0, vocab_size, (4, 32)).to(device)

logits = model(x)
loss = loss_fn(logits.view(-1, vocab_size), y.view(-1))

print(f"Loss: {loss.item():.4f}")

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("PASSED: Backward pass works!")

TEST 5: Backward Pass
--------------------------------------------------
Loss: 4.7701
PASSED: Backward pass works!


---

## Overfit

In [14]:
print("TEST 6: Overfit Test (Memorize 1 Batch)")
print("-" * 50)

model = GPT(vocab_size=vocab_size, d_model=64, num_heads=4, num_layers=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

x = torch.randint(0, vocab_size, (2, 16)).to(device)
y = torch.randint(0, vocab_size, (2, 16)).to(device)

losses = []
for step in range(50):
    model.train()
    logits = model(x)
    loss = loss_fn(logits.view(-1, vocab_size), y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if step % 10 == 0:
        print(f"  Step {step}: {loss.item():.4f}")

print(f"\nLoss decay: {losses[0]:.4f} → {losses[-1]:.4f}")
decay_ratio = losses[-1] / losses[0]

if decay_ratio < 0.5:
    print(f"PASSED: Model learned (decay ratio: {decay_ratio:.2f})")
else:
    print(f"FAILED: Model didn't learn (decay ratio: {decay_ratio:.2f})")
    print("  Check: architecture, gradients, learning rate")

TEST 6: Overfit Test (Memorize 1 Batch)
--------------------------------------------------
  Step 0: 4.8213
  Step 10: 0.0649
  Step 20: 0.0055
  Step 30: 0.0018
  Step 40: 0.0010

Loss decay: 4.8213 → 0.0006
PASSED: Model learned (decay ratio: 0.00)


---

## Training loop

In [15]:
print("TEST 7: Full Training Loop (5 iterations)")
print("-" * 50)

model = GPT(vocab_size=vocab_size, d_model=64, num_heads=4, num_layers=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

def get_batch(ids, batch_size, seq_len, device):
    max_start = len(ids) - seq_len
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([ids[i:i+seq_len] for i in starts])
    y = torch.stack([ids[i+1:i+seq_len+1] for i in starts])
    return x.to(device), y.to(device)

train_ids = torch.randint(0, vocab_size, (1000,))

for epoch in range(5):
    model.train()
    total_loss = 0

    for step in range(10):
        x, y = get_batch(train_ids, batch_size=16, seq_len=32, device=device)

        logits = model(x)
        loss = loss_fn(logits.view(-1, vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / 10
    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")

print("PASSED: Full training loop works!")

TEST 7: Full Training Loop (5 iterations)
--------------------------------------------------
Epoch 1: Loss = 4.6727
Epoch 2: Loss = 4.4779
Epoch 3: Loss = 4.3224
Epoch 4: Loss = 4.1466
Epoch 5: Loss = 3.9896
PASSED: Full training loop works!


---

# Demo

---

## Load model and generate

In [23]:
model = GPT(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    num_heads=4,
    num_layers=4,
    dropout=0.1
).to(device)
model.load_state_dict(torch.load("gpt_model.pt"))
model.eval()

def generate(text, max_length = 50, temperature = 1.0):
    model.eval()
    input_ids = tokenizer.encode(text).unsqueeze(0).to(device)

    generated = input_ids.clone()

    with torch.no_grad():
        for _ in range(max_length):
            outputs = model(generated)

            next_token_logits = outputs[0, -1, :] / temperature

            probs = torch.softmax(next_token_logits, dim = -1)
            next_token = torch.multinomial(probs, num_samples = 1)

            generated = torch.cat([generated, next_token.unsqueeze(0)], dim = 1)

    return tokenizer.decode(generated[0])

---

## UI

In [24]:
import gradio as gr

def generate_ui(text):
    return generate(text)

def continue_text(text):
    return generate(text)

def submit_feedback(feedback_text):
    print(f"Feedback received: {feedback_text}")
    return "Thank you for your feedback!"

with gr.Blocks() as demo:
    gr.Markdown("# GPT Text Generator")
    with gr.Tab("Generate Text"):
        inp = gr.Textbox(label = "Prompt", placeholder = "Start typing your prompt here...")
        out = gr.Textbox(label = "Generated Text")

        gr.Button("Generate").click(generate_ui, inp, out)
        gr.Button("Continue from output").click(continue_text, out, out)

    with gr.Tab("Feedback"):
        feedback_input = gr.Textbox(label = "Your Feedback", placeholder = "Tell us what you think...")
        feedback_output = gr.Textbox(label = "Status", interactive = False)
        gr.Button("Submit Feedback").click(submit_feedback, feedback_input, feedback_output)

demo.launch(share = True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dfbfbca519de8a68fb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:
print("Testing Gradio UI with a sample prompt...")
sample_prompt = "The quick brown fox jumped over the lazy dog"
generated_text_from_ui = generate_ui(sample_prompt)
print(f"\nPrompt: '{sample_prompt}'")
print(f"Generated Text (simulated UI output):\n{generated_text_from_ui}")

Testing Gradio UI with a sample prompt...

Prompt: 'The quick brown fox jumped over the lazy dog'
Generated Text (simulated UI output):
The quick brown fox jumped over the lazy dogs.

The room without care he right in his chyste


In [27]:
print("Simulating Gradio UI text generation with a new prompt...")
new_sample_prompt = "The sun set over the horizon"
新たに出力されたテキスト_UI = generate_ui(new_sample_prompt)
print(f"\nPrompt: '{new_sample_prompt}'")
print(f"Generated Text (simulated UI output):\n{新たに出力されたテキスト_UI}")

Simulating Gradio UI text generation with a new prompt...

Prompt: 'The sun set over the horizon'
Generated Text (simulated UI output):
The sun set over the horizon MaBsh Throyanic plasage composes,
On those who h


# Task
Load the trained GPT model from "gpt_model.pt" and generate text. Then, present the generated text to the user and ask if they would like to generate more text or try different prompts.

## Load Model

### Subtask:
Load the trained GPT model from the 'gpt_model.pt' file. This will instantiate the GPT class with the correct architecture and load the saved weights.


The instructions for loading the model are already implemented in the code cell `DeJCcwGnvgLX` under the 'Text generation' section. This cell instantiates the `GPT` model with the correct architecture, loads its state from `gpt_model.pt`, and sets it to evaluation mode (`model.eval()`).

```python
model = GPT(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    num_heads=4,
    num_layers=4,
    dropout=0.1
).to(device)
model.load_state_dict(torch.load("gpt_model.pt"))
model.eval()
```

Therefore, no further action is required for this subtask.

## Generate Text

### Subtask:
Generate a sample text using the loaded model and a default prompt.


**Reasoning**:
The subtask is to generate a sample text using the loaded model and a default prompt. The instructions specify to use the `generate_ui` function with a prompt. I will call `generate_ui` with a sample prompt and print the result.



In [28]:
print('Generating sample text with a default prompt...')
default_prompt = "Hello, my name is"
generated_text = generate_ui(default_prompt)
print(f"\nPrompt: '{default_prompt}'")
print(f"Generated Text:\n{generated_text}")

Generating sample text with a default prompt...

Prompt: 'Hello, my name is'
Generated Text:
Hello, my name island for head. I stayed as it is no fumality
word


Here is the generated text:

```
Prompt: 'Hello, my name is'
Generated Text:
Hello, my name island for head. I stayed as it is no fumality
word
```

Would you like to:
* Generate more text with a different prompt?
* Generate more text by continuing from the output?
* Try a different feature or explore the model further?

Here is the generated text:

```
Prompt: 'Hello, my name is'
Generated Text:
Hello, my name island for head. I stayed as it is no fumality
word
```

Would you like to:
* Generate more text with a different prompt?
* Generate more text by continuing from the output?
* Try a different feature or explore the model further?

Here is the generated text:

```
Prompt: 'Hello, my name is'
Generated Text:
Hello, my name island for head. I stayed as it is no fumality
word
```

Would you like to:
* Generate more text with a different prompt?
* Generate more text by continuing from the output?
* Try a different feature or explore the model further?

Here is the generated text:

```
Prompt: 'Hello, my name is'
Generated Text:
Hello, my name island for head. I stayed as it is no fumality
word
```

Would you like to:
* Generate more text with a different prompt?
* Generate more text by continuing from the output?
* Try a different feature or explore the model further?

## Final Task

### Subtask:
Present the generated text to the user and ask if they would like to generate more text or try different prompts.


## Summary:

### Data Analysis Key Findings
*   The GPT model was successfully loaded from the `gpt_model.pt` file. This involved instantiating the `GPT` class with a `vocab_size`, `d_model=128`, `num_heads=4`, `num_layers=4`, and `dropout=0.1`, then loading its saved state dictionary and setting it to evaluation mode.
*   A sample text was generated using the default prompt "Hello, my name is".
*   The generated text output was: "Hello, my name island for head. I stayed as it is no fumality\nword".
*   The generated text was presented to the user, along with options to generate more text with a different prompt, continue from the output, or explore the model further.

### Insights or Next Steps
*   The model successfully performs text generation based on a given prompt, but the quality of the output suggests potential for improvement, possibly through additional training or more complex prompt engineering.
*   The system is now ready to receive user input for generating more text or trying different prompts, allowing for interactive exploration of the model's capabilities.
